In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
class RNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, 64, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(64, 64)
        self.out = nn.Linear(64, 1)
        self.act = nn.ReLU()
    
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class WeightedL1(nn.Module):
    def forward(self, pred, target):
        weights = 1.0 + 1.0 * torch.clamp(torch.abs(target) - 2.0, min=0)
        return (torch.abs(pred - target) * weights).mean()


def predict(model, loader):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(x.to(device)).cpu().numpy() for x, _ in loader]).flatten()

print("✅ Components ready")

✅ Components ready


In [3]:
# Load data
print("Loading data...")
train_loader, val_loader, pipeline = load_data(
    train_path="../datasets_challenge/train",
    train_ratio=0.98,
    batchsize=64,
    columns_to_drop=["sideSlip", "vyCG", "time", "longitude", "latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

test_loader = load_test_data(
    test_path="../datasets_challenge/test",
    batchsize=512,
    pipeline=pipeline,
    load_anonymized_data=True
)

X_sample, _ = next(iter(train_loader))
input_size = X_sample.shape[2]
print(f"✅ Data loaded | Input size: {input_size}")

Loading data...
../datasets_challenge/train/train_part_001.csv
../datasets_challenge/train/train_part_002.csv
../datasets_challenge/train/train_part_003.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.csv
../datasets_challenge/train/train_part_014.csv
../datasets_challenge/train/train_part_015.c

## 💡 Idea 1: Three-Stage Training

Current best is 2-stage. What if we add a 3rd stage with even lower LR?

In [4]:
def train_three_stage(seed, verbose=True):
    """
    Three-stage training:
    - Stage 1: L1, lr=0.0003, 30 epochs (build foundation)
    - Stage 2: WeightedL1, lr=0.00005, 30 epochs (few-shot focus)
    - Stage 3: WeightedL1, lr=0.00001, 15 epochs (fine polish)
    """
    seed_everything(seed)
    model = RNN(input_size).to(device)
    crit_l1 = nn.L1Loss()
    crit_w = WeightedL1()
    
    # Stage 1
    if verbose: print("  S1: L1, lr=3e-4, 30ep", end="")
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    best, state = np.inf, None
    for ep in range(30):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit_l1(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit_l1(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    if verbose: print(f" → {best:.5f}")
    
    # Stage 2
    if verbose: print("  S2: WeightedL1, lr=5e-5, 30ep", end="")
    opt = torch.optim.Adam(model.parameters(), lr=0.00005)
    best2, state2 = np.inf, None
    for ep in range(30):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit_w(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit_l1(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    if verbose: print(f" → {best2:.5f}")
    
    # Stage 3 (NEW!)
    if verbose: print("  S3: WeightedL1, lr=1e-5, 15ep", end="")
    opt = torch.optim.Adam(model.parameters(), lr=0.00001)
    best3, state3 = np.inf, None
    for ep in range(15):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit_w(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit_l1(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best3: best3, state3 = val, model.state_dict().copy()
    model.load_state_dict(state3)
    if verbose: print(f" → {best3:.5f}")
    
    return model, best3

print("✅ Three-stage training function ready")

✅ Three-stage training function ready


## 💡 Idea 2: Cosine Annealing LR (Single Stage 2)

In [5]:
def train_cosine_lr(seed, verbose=True):
    """
    Two-stage with cosine annealing in Stage 2:
    - Stage 1: L1, lr=0.0003, 30 epochs
    - Stage 2: WeightedL1, lr=5e-5→1e-6 (cosine), 50 epochs
    """
    seed_everything(seed)
    model = RNN(input_size).to(device)
    crit_l1 = nn.L1Loss()
    crit_w = WeightedL1()
    
    # Stage 1
    if verbose: print("  S1: L1, lr=3e-4, 30ep", end="")
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    best, state = np.inf, None
    for ep in range(30):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit_l1(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit_l1(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    if verbose: print(f" → {best:.5f}")
    
    # Stage 2 with Cosine LR
    if verbose: print("  S2: WeightedL1, cosine lr 5e-5→1e-6, 50ep", end="")
    opt = torch.optim.Adam(model.parameters(), lr=0.00005)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50, eta_min=1e-6)
    best2, state2 = np.inf, None
    for ep in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit_w(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        scheduler.step()
        model.eval()
        val = sum(crit_l1(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    if verbose: print(f" → {best2:.5f}")
    
    return model, best2

print("✅ Cosine LR training function ready")

✅ Cosine LR training function ready


## 💡 Idea 3: Longer Stage 1 (40 epochs)

In [6]:
def train_long_stage1(seed, verbose=True):
    """
    Longer Stage 1:
    - Stage 1: L1, lr=0.0003, 40 epochs (longer foundation)
    - Stage 2: WeightedL1, lr=5e-5, 40 epochs
    """
    seed_everything(seed)
    model = RNN(input_size).to(device)
    crit_l1 = nn.L1Loss()
    crit_w = WeightedL1()
    
    # Stage 1 (LONGER)
    if verbose: print("  S1: L1, lr=3e-4, 40ep", end="")
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    best, state = np.inf, None
    for ep in range(40):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit_l1(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit_l1(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    if verbose: print(f" → {best:.5f}")
    
    # Stage 2
    if verbose: print("  S2: WeightedL1, lr=5e-5, 40ep", end="")
    opt = torch.optim.Adam(model.parameters(), lr=0.00005)
    best2, state2 = np.inf, None
    for ep in range(40):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit_w(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit_l1(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    if verbose: print(f" → {best2:.5f}")
    
    return model, best2

print("✅ Long Stage 1 training function ready")

✅ Long Stage 1 training function ready


In [7]:
def train_baseline(seed):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    print(f"  Baseline L1", end="")
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state, patience = np.inf, None, 0
    for ep in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: 
            best, state, patience = val, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10: break
    model.load_state_dict(state)
    print(f" → {best:.5f}")
    return model, best

## 🚀 Run All Ideas

In [8]:
SEEDS = [42, 2024]
results = {}

# Baseline
print("\n📦 BASELINE")
baseline_preds = []
for seed in SEEDS:
    print(f"🌱 Seed {seed}")
    model, _ = train_baseline(seed)
    baseline_preds.append(predict(model, test_loader))
baseline_ensemble = np.mean(baseline_preds, axis=0)
print("✅ Baseline ready")


📦 BASELINE
🌱 Seed 42
  Baseline L1  Baseline L1 → 0.11366
 → 0.11366
🌱 Seed 2024
  Baseline L1🌱 Seed 2024
  Baseline L1 → 0.11157
 → 0.11157
✅ Baseline ready
✅ Baseline ready


In [9]:
# Idea 1: Three-Stage
print("\n" + "="*50)
print("📦 THREE-STAGE TRAINING")
print("="*50)
preds = []
for seed in SEEDS:
    print(f"🌱 Seed {seed}")
    model, _ = train_three_stage(seed)
    preds.append(predict(model, test_loader))
results["three_stage"] = np.mean(preds, axis=0)
print("✅ Done")


📦 THREE-STAGE TRAINING
🌱 Seed 42
  S1: L1, lr=3e-4, 30ep → 0.12387
  S2: WeightedL1, lr=5e-5, 30ep → 0.12387
  S2: WeightedL1, lr=5e-5, 30ep → 0.11336
  S3: WeightedL1, lr=1e-5, 15ep → 0.11336
  S3: WeightedL1, lr=1e-5, 15ep → 0.11187
 → 0.11187
🌱 Seed 2024
  S1: L1, lr=3e-4, 30ep🌱 Seed 2024
  S1: L1, lr=3e-4, 30ep → 0.12372
  S2: WeightedL1, lr=5e-5, 30ep → 0.12372
  S2: WeightedL1, lr=5e-5, 30ep → 0.11291
  S3: WeightedL1, lr=1e-5, 15ep → 0.11291
  S3: WeightedL1, lr=1e-5, 15ep → 0.11129
 → 0.11129
✅ Done
✅ Done


In [10]:
# Idea 2: Cosine LR
print("\n" + "="*50)
print("📦 COSINE ANNEALING LR")
print("="*50)
preds = []
for seed in SEEDS:
    print(f"🌱 Seed {seed}")
    model, _ = train_cosine_lr(seed)
    preds.append(predict(model, test_loader))
results["cosine_lr"] = np.mean(preds, axis=0)
print("✅ Done")


📦 COSINE ANNEALING LR
🌱 Seed 42
  S1: L1, lr=3e-4, 30ep → 0.12630
  S2: WeightedL1, cosine lr 5e-5→1e-6, 50ep → 0.12630
  S2: WeightedL1, cosine lr 5e-5→1e-6, 50ep → 0.11284
 → 0.11284
🌱 Seed 2024
  S1: L1, lr=3e-4, 30ep🌱 Seed 2024
  S1: L1, lr=3e-4, 30ep → 0.12364
  S2: WeightedL1, cosine lr 5e-5→1e-6, 50ep → 0.12364
  S2: WeightedL1, cosine lr 5e-5→1e-6, 50ep → 0.11286
 → 0.11286
✅ Done
✅ Done


In [11]:
# Idea 3: Long Stage 1
print("\n" + "="*50)
print("📦 LONG STAGE 1 (40 epochs)")
print("="*50)
preds = []
for seed in SEEDS:
    print(f"🌱 Seed {seed}")
    model, _ = train_long_stage1(seed)
    preds.append(predict(model, test_loader))
results["long_s1"] = np.mean(preds, axis=0)
print("✅ Done")


📦 LONG STAGE 1 (40 epochs)
🌱 Seed 42
  S1: L1, lr=3e-4, 40ep → 0.11800
  S2: WeightedL1, lr=5e-5, 40ep → 0.11800
  S2: WeightedL1, lr=5e-5, 40ep → 0.10749
 → 0.10749
🌱 Seed 2024
  S1: L1, lr=3e-4, 40ep🌱 Seed 2024
  S1: L1, lr=3e-4, 40ep → 0.11616
  S2: WeightedL1, lr=5e-5, 40ep → 0.11616
  S2: WeightedL1, lr=5e-5, 40ep → 0.10830
 → 0.10830
✅ Done
✅ Done


In [12]:
# Save all predictions
print("\n" + "="*60)
print("💾 SAVING (50/50 blend)")
print("="*60)

for name, ens in results.items():
    blend = (baseline_ensemble + ens) / 2
    filename = f"pred_v8b_{name}.csv"
    pd.DataFrame({"id": range(len(blend)), "predicted_sideslip": blend}).to_csv(filename, index=False)
    print(f"✅ {filename}")


💾 SAVING (50/50 blend)
✅ pred_v8b_three_stage.csv
✅ pred_v8b_cosine_lr.csv
✅ pred_v8b_long_s1.csv


In [13]:
print("\n" + "="*60)
print("📋 SUMMARY")
print("="*60)

print("\n🏆 Current Best: 0.21751")

print("\n📁 Alternative Ideas:")
print("  - pred_v8b_three_stage.csv  (S1→S2→S3 with decaying LR)")
print("  - pred_v8b_cosine_lr.csv    (cosine annealing 5e-5→1e-6)")
print("  - pred_v8b_long_s1.csv      (40ep Stage 1)")

print("\n💡 Hypothesis:")
print("  - 3-stage might help if model benefits from gradual refinement")
print("  - Cosine LR is proven in many settings")
print("  - Longer Stage 1 = better foundation = better few-shot?")


📋 SUMMARY

🏆 Current Best: 0.21751

📁 Alternative Ideas:
  - pred_v8b_three_stage.csv  (S1→S2→S3 with decaying LR)
  - pred_v8b_cosine_lr.csv    (cosine annealing 5e-5→1e-6)
  - pred_v8b_long_s1.csv      (40ep Stage 1)

💡 Hypothesis:
  - 3-stage might help if model benefits from gradual refinement
  - Cosine LR is proven in many settings
  - Longer Stage 1 = better foundation = better few-shot?
